# 03 · Collect frames from your own footage for labelling

The models score 0.94 (detector) and 0.995 (court) on their Roboflow validation sets, yet on a real
NBA broadcast clip they found 5.9 of 10 players and 5.2 of 18 court keypoints per frame. Validation
images come from the same datasets as training, so they cannot show this gap. The fix is labelled
frames from the footage the pipeline actually runs on, and the cheapest way to get them is to label
the frames the current models get wrong.

This notebook runs both models over your clips, scores every sampled frame by how much the models
struggled (missing players, no ball or several balls, low-confidence boxes, too few court keypoints),
and exports the hardest frames, spread out in time, with the models' predictions as pre-labels. In
Roboflow you then only correct boxes instead of drawing them.

**Loop:** run this, upload `detector/` and `court/` to two Roboflow projects, correct the labels,
download them in YOLO format to `MyDrive/courtvision/training_data/own_labelled` (detector) and
`.../own_labelled_court` (court), retrain with notebooks 01 and 02, repeat. 150-300 corrected frames
per round is a good first target.

Keep a few whole clips aside as a **test set** that you never export frames from, and label their
touches by hand: that is the only way to score the touch table itself.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
!git clone -q {REPO_URL} /content/courtvision || (cd /content/courtvision && git pull -q)
!pip -q install -e "/content/courtvision[models]"

## Settings

`TEST_CLIPS` are skipped so they stay unseen. `STRIDE` samples one frame in N (every 10th frame of
30 fps video is 3 per second). `MIN_GAP` keeps chosen frames at least that many frames apart, so a
single hard moment does not fill the whole budget.

In [ ]:
import glob
CLIPS = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
TEST_CLIPS = []                       # file names to hold out, e.g. ["lebron_block.mp4"]
CLIPS = [c for c in CLIPS if os.path.basename(c) not in TEST_CLIPS]
FRAMES_PER_CLIP = 40
STRIDE = 10
MIN_GAP = 45                          # frames, 1.5 s at 30 fps
LOW_CONF, KEEP_CONF = 0.10, 0.25      # boxes in between count as "uncertain"; pre-labels use KEEP_CONF
KP_CONF = 0.5
OUT = f"{DRIVE_ROOT}/training_data/to_label"
print(len(CLIPS), "clips:", [os.path.basename(c) for c in CLIPS])

## Score frames

In [ ]:
import cv2, numpy as np
from ultralytics import YOLO
from courtvision.training.mining import FrameStats, pick_frames

det = YOLO(f"{DRIVE_ROOT}/models/player_detector.pt")
court = YOLO(f"{DRIVE_ROOT}/models/court_keypoints.pt")
cls_id = {n: i for i, n in det.names.items()}
PLAYER, BALL = cls_id["Player"], cls_id["Ball"]

def score_clip(path):
    cap = cv2.VideoCapture(path)
    stats, f = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if f % STRIDE == 0:
            b = det.predict(frame, imgsz=1280, conf=LOW_CONF, verbose=False)[0].boxes
            conf, cls = b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)
            keep = conf >= KEEP_CONF
            kp = court.predict(frame, imgsz=640, verbose=False)[0].keypoints
            n_kp = 0 if kp is None or kp.conf is None or len(kp.conf) == 0 else int((kp.conf[0].cpu().numpy() >= KP_CONF).sum())
            stats.append(FrameStats(frame=f, players=int((keep & (cls == PLAYER)).sum()),
                                    balls=int((keep & (cls == BALL)).sum()), uncertain=int((~keep).sum()), keypoints=n_kp))
        f += 1
    cap.release()
    return stats

all_stats = {c: score_clip(c) for c in CLIPS}
for c, s in all_stats.items():
    print(f"{os.path.basename(c):30s} frames={len(s):4d}  mean players={np.mean([x.players for x in s]):.1f}"
          f"  no-ball={np.mean([x.balls == 0 for x in s]):.0%}  mean keypoints={np.mean([x.keypoints for x in s]):.1f}")

## Export the hardest frames with pre-labels

Frames where the court model sees no keypoints at all are usually replays, close-ups or crowd shots.
They go to `negative_candidates/`: look through them and move the ones with no players or ball in play
into `training_data/negatives`, which notebook 01 adds as hard negatives.

In [ ]:
import yaml
from courtvision.training.mining import pose_row, yolo_rows

for sub in ["detector/images", "detector/labels", "court/images", "court/labels", "negative_candidates"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

def read_frame(path, idx):
    cap = cv2.VideoCapture(path); cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read(); cap.release()
    return frame if ok else None

n_det = n_court = n_neg = 0
for clip, stats in all_stats.items():
    stem = os.path.splitext(os.path.basename(clip))[0]
    by_frame = {s.frame: s for s in stats}
    for idx in pick_frames(stats, FRAMES_PER_CLIP, min_gap=MIN_GAP):
        frame = read_frame(clip, idx)
        if frame is None:
            continue
        name = f"{stem}_f{idx:06d}"
        H, W = frame.shape[:2]
        if by_frame[idx].keypoints == 0:
            cv2.imwrite(f"{OUT}/negative_candidates/{name}.jpg", frame); n_neg += 1
            continue
        b = det.predict(frame, imgsz=1280, conf=KEEP_CONF, verbose=False)[0].boxes
        cv2.imwrite(f"{OUT}/detector/images/{name}.jpg", frame)
        rows = yolo_rows(b.xyxy.cpu().numpy(), b.cls.cpu().numpy(), (W, H))
        open(f"{OUT}/detector/labels/{name}.txt", "w").write("\n".join(rows) + ("\n" if rows else "")); n_det += 1
        if by_frame[idx].keypoints < 6:
            kp = court.predict(frame, imgsz=640, verbose=False)[0].keypoints
            row = None
            if kp is not None and kp.conf is not None and len(kp.conf):
                row = pose_row(kp.xy[0].cpu().numpy(), kp.conf[0].cpu().numpy(), (W, H), conf=KP_CONF)
            cv2.imwrite(f"{OUT}/court/images/{name}.jpg", frame)
            open(f"{OUT}/court/labels/{name}.txt", "w").write((row + "\n") if row else ""); n_court += 1

yaml.safe_dump({"names": dict(det.names)}, open(f"{OUT}/detector/data.yaml", "w"))
yaml.safe_dump({"names": {0: "court"}, "kpt_shape": [18, 3]}, open(f"{OUT}/court/data.yaml", "w"))
print(f"detector frames: {n_det}  court frames: {n_court}  negative candidates: {n_neg}  ->  {OUT}")

In [ ]:
# Zip each folder for Roboflow's upload page (drag the zip in, it reads the YOLO labels as pre-labels).
import shutil
for sub in ["detector", "court"]:
    print(shutil.make_archive(f"{OUT}/{sub}", "zip", f"{OUT}/{sub}"))
# Roboflow reads the detector pre-labels directly. A keypoint project needs its 18 keypoints defined
# first (same order as notebook 02); if it does not pick up the court pre-labels, label those frames
# from scratch, which is only 4-8 points per frame.